# S0E1 lab · Small errors don't stay small

**Video:** [Why a Robot Isn't a Chatbot With Arms](https://youtu.be/bN-Yv_gUcMM) (the H2 scene, "off by 21 mm at step 100")
**Book:** [Action Models for Robot Learning, §1.1](https://action-models-book.vercel.app/chapters/1/1/)
**You need:** Python basics. No GPU. About 20 minutes.

In the video, FMR-6 tries to draw a straight 200 mm line in 100 steps of 2 mm, and every step is off by just 0.12°. Here you'll run that yourself, watch the arm do it in MuJoCo, and then fix it with one line of feedback.

In [ ]:
# Setup: works in Google Colab and locally (after `pip install -e .` in the repo root)
import os, sys
if "google.colab" in sys.modules:
    os.environ["MUJOCO_GL"] = "egl"
    if not os.path.exists("fmr-robot-learning-labs"):
        !git clone -q https://github.com/kandapagari/fmr-robot-learning-labs.git
    %cd fmr-robot-learning-labs
    !pip install -q -e .
    sys.path.insert(0, "labs/s0e1-drift")
else:
    sys.path.insert(0, os.path.abspath("."))
import numpy as np, matplotlib.pyplot as plt, mediapy as media
import fmr6, drift

## 1 · Meet FMR-6

The same six-axis arm you see in the videos: base yaw, shoulder, elbow, then a spherical wrist (J4, J5, J6 meet at one point) and a two-finger gripper.

In [ ]:
robot = fmr6.Robot("table")
pos, R = robot.tcp()
print("tool tip (m):", pos.round(3), " tool axis:", R[:, 0].round(2))
media.show_image(robot.render(480, 360))

## 2 · The policy and its tiny mistake

Each step the "policy" picks a heading for the next 2 mm. Its heading is biased by 0.12°, and because each step starts where the last one ended, the bias accumulates. Read `drift.open_loop` (it's 8 lines), then run it.

In [ ]:
ol = drift.open_loop(steps=100, step_mm=2.0, bias_deg=0.12)
for k in (1, 10, 50, 100):
    print(f"step {k:3d}: off the line by {drift.offset_at(ol, k):7.3f} mm")

At step 1 the error is four thousandths of a millimetre. Nobody would notice it in a demonstration. By step 100 the tool is about 2 cm off.

## 3 · Watch FMR-6 do it

We put the path on the table and use inverse kinematics to move the tool tip along it, tool pointing down.

In [ ]:
path = drift.to_world(ol)                     # metres, on the table in front of the robot
frames, q = [], None
for k in range(0, len(path), 4):
    q = robot.ik(path[k], tool_dir=(0, 0, -1), q0=q)
    frames.append(robot.render(320, 240, camera={"lookat": [0.62, 0.0, 0.40], "distance": 1.0, "azimuth": 180, "elevation": -60}))
media.show_video(frames, fps=12)

## 4 · One line of feedback

A policy that looks at where it actually is can steer back. `drift.closed_loop` adds `- gain * y / step_mm` to the heading. Compare:

In [ ]:
cl = drift.closed_loop(gain=0.05)
plt.figure(figsize=(7, 3))
plt.plot(ol[:, 0], ol[:, 1], label="open loop (replays the plan)")
plt.plot(cl[:, 0], cl[:, 1], label="closed loop (looks, then corrects)")
plt.axhline(0, color="k", lw=0.8, ls="--", label="planned line")
plt.xlabel("along the line (mm)"); plt.ylabel("sideways (mm)"); plt.legend(); plt.tight_layout(); plt.show()
print(f"step 100: open loop {drift.offset_at(ol, 100):.1f} mm, closed loop {drift.offset_at(cl, 100):.2f} mm")

## 5 · Try it

1. Halve the bias to 0.06°. Does the error at step 100 halve too? Why?
2. Keep the bias and double the number of steps (same 2 mm each). How far off is step 200?
3. Raise `gain` until the closed-loop path starts to wiggle. What's happening?
4. **The robot-learning point:** a demonstration is recorded by someone doing the task correctly, so it never shows what to do *after* a small mistake. Which of the two policies above is behaviour cloning on such demos more like? (Season 2 covers the fix: DAgger.)

In [ ]:
# your experiments here